# California Housing Price Prediction - Model Experimentation

This notebook focuses on:
1. Loading and preparing the processed data
2. Training different models
3. Hyperparameter tuning using GridSearchCV
4. Model evaluation and selection

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Load autoreload extension
%load_ext autoreload
%autoreload 2

import sys
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso,ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

# Add the src directory to path
sys.path.append('../src')
from data.data_processor import DataProcessor

## 1. Load and Prepare Data

In [3]:
# Initialize data processor
data_processor = DataProcessor()
# Load processed data
df = data_processor.load_data()

In [4]:
# fill null values with mean
df['total_bedrooms'].fillna(df['total_bedrooms'].mean(), inplace=True)

In [7]:
def split_data(df, target_column, test_size=0.2, random_state=42):
    # Separate features and target
    X = df.drop(target_column, axis=1)
    y = df[target_column]
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def encode_categorical_columns(X_train, X_test, categorical_columns):
    one_hot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    
    # Fit the encoder on training data and transform
    X_train_encoded_categorical = one_hot_encoder.fit_transform(X_train[categorical_columns])
    X_train_encoded_categorical = pd.DataFrame(
        X_train_encoded_categorical, 
        columns=one_hot_encoder.get_feature_names_out(categorical_columns),
        index=X_train.index
    )
    
    # Transform the test data using the same encoder
    X_test_encoded_categorical = one_hot_encoder.transform(X_test[categorical_columns])
    X_test_encoded_categorical = pd.DataFrame(
        X_test_encoded_categorical, 
        columns=one_hot_encoder.get_feature_names_out(categorical_columns),
        index=X_test.index
    )
    
    # Save the encoder for future inference
    
    os.makedirs('../models/v1', exist_ok=True)
    with open('../models/v1/one_hot_encoder.pkl', 'wb') as f:
        pickle.dump(one_hot_encoder, f)
    
    return X_train_encoded_categorical, X_test_encoded_categorical

def scale_numeric_columns(X_train, X_test, numeric_columns):
    scaler = StandardScaler()
    X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])
    X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])
    
    # Save the scaler for future inference
    import pickle
    with open('../models/v1/scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    
    return X_train, X_test, scaler

def preprocess_data(df, target_column, test_size=0.2, random_state=42):
    # Split the data
    X_train, X_test, y_train, y_test = split_data(df, target_column, test_size, random_state)
    
    # Identify categorical and numerical columns
    categorical_columns = X_train.select_dtypes(include=['object']).columns
    numeric_columns = X_train.select_dtypes(exclude=['object']).columns
    
    # Encode categorical columns
    X_train_encoded_categorical, X_test_encoded_categorical = encode_categorical_columns(
        X_train, X_test, categorical_columns
    )
    
    # Combine encoded categorical columns with numerical columns
    X_train_encoded = pd.concat([X_train.drop(categorical_columns, axis=1), X_train_encoded_categorical], axis=1)
    X_test_encoded = pd.concat([X_test.drop(categorical_columns, axis=1), X_test_encoded_categorical], axis=1)
    
    # Align train and test sets to ensure they have the same columns
    X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)
    
    # Scale numerical columns
    X_train_encoded, X_test_encoded, scaler = scale_numeric_columns(X_train_encoded, X_test_encoded, numeric_columns)
    
    return X_train_encoded, X_test_encoded, y_train, y_test, scaler


In [8]:

# Call the function
X_train_processed, X_test_processed, y_train, y_test, scaler = preprocess_data(df, target_column='target')


In [9]:
X_train_processed.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
14196,1.272587,-1.372811,0.348490,0.222569,0.211228,0.768276,0.322906,-0.326196,0.0,0.0,0.0,0.0,1.0
8267,0.709162,-0.876696,1.618118,0.340293,0.593094,-0.098901,0.672027,-0.035843,0.0,0.0,0.0,0.0,1.0
17445,-0.447603,-0.460146,-1.952710,-0.342597,-0.495226,-0.449818,-0.430461,0.144701,0.0,0.0,0.0,0.0,1.0
14265,1.232698,-1.382172,0.586545,-0.561490,-0.409306,-0.007434,-0.380587,-1.017864,0.0,0.0,0.0,0.0,1.0
2271,-0.108551,0.532084,1.142008,-0.119565,-0.256559,-0.485877,-0.314962,-0.171488,0.0,1.0,0.0,0.0,0.0


## 2. Model Training and Evaluation Function

In [12]:
def evaluate_model(model, X_train, X_test, y_train, y_test):
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Perform cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    return {
        'RMSE': rmse,
        'R2': r2,
        'CV Mean R2': cv_scores.mean(),
        'CV Std R2': cv_scores.std()
    }

## 3. Basic Model Comparison

In [13]:
# Initialize models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

# Evaluate each model
results = {}

for name, model in models.items():
    print(f"Evaluating {name}...")
    results[name] = evaluate_model(model, X_train_processed, X_test_processed, y_train, y_test)

# Create results DataFrame
results_df = pd.DataFrame(results).T
results_df

Evaluating Linear Regression...
Evaluating Ridge...
Evaluating Lasso...
Evaluating Decision Tree...
Evaluating Random Forest...
Evaluating Gradient Boosting...


,RMSE,R2,CV Mean R2,CV Std R2
Linear Regression,70031.419920,0.625735,0.647542,0.011454
Ridge,70038.344061,0.625661,0.647552,0.011438
Lasso,70032.566823,0.625723,0.647542,0.011449
Decision Tree,69254.033288,0.633998,0.636706,0.007572
Random Forest,48925.828360,0.817329,0.818346,0.003690
Gradient Boosting,55963.617174,0.760996,0.773336,0.002396


## 4. Hyperparameter Tuning

Based on the initial results, we'll perform hyperparameter tuning on the top performing models.

In [15]:
# Hyperparameter grids
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [10, 20, 30, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4]
    },
    'Gradient Boosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.1, 0.3],
        'max_depth': [3, 4, 5],
        'min_samples_split': [2, 5]
    },
    'Ridge': {
        'alpha': [0.1, 1.0, 10.0, 100.0]
    }
}

In [16]:
# Perform GridSearch
best_models = {}
for name, param_grid in param_grids.items():
    print(f"\nTuning {name}...")
    model = models[name]
    grid_search = GridSearchCV(
        model,
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1)
    grid_search.fit(X_train_processed, y_train)
    
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
    
    best_models[name] = grid_search.best_estimator_


Tuning Random Forest...
Best parameters: {'max_depth': 30, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 300}
Best cross-validation score: 0.8203

Tuning Gradient Boosting...
Best parameters: {'learning_rate': 0.1, 'max_depth': 5, 'min_samples_split': 5, 'n_estimators': 300}
Best cross-validation score: 0.8294

Tuning Ridge...
Best parameters: {'alpha': 1.0}
Best cross-validation score: 0.6476


## 5. Final Model Evaluation

In [ ]:
# Evaluate tuned models
tuned_results = {}
for name, model in best_models.items():
    print(f"Evaluating tuned {name}...")
    tuned_results[name] = evaluate_model(model, X_train_processed, X_test_processed, y_train, y_test)

# Create tuned results DataFrame
tuned_results_df = pd.DataFrame(tuned_results).T
tuned_results_df

## 6. Save Best Model

After comparing the results, we'll save the best performing model for future use.

In [ ]:
import joblib

# Get the best model based on R2 score
best_model_name = tuned_results_df['R2'].idxmax()
best_model = best_models[best_model_name]

# Save the model and scaler
os.makedirs('../models/v1', exist_ok=True)
joblib.dump(best_model, '../models/v1/best_model.joblib')
joblib.dump(scaler, '../models/scaler.joblib')

print(f"Best model ({best_model_name}) saved successfully!")